# Load Silver Tables

**Purpose:** Load cleaned data from Silver layer (lakehouse tables)

---

## Read tables silver


In [1]:
from pyspark.sql.functions import *
from pyspark.sql import functions as F

# Load Silver tables
df_customers_clean = spark.table("silver_customers")
df_policies_clean = spark.table("silver_policies")
df_claims_clean = spark.table("silver_claims")

print("Silver tables loaded:")
print(f" - Customers: {df_customers_clean.count()} rows")
print(f" - Policies: {df_policies_clean.count()} rows")
print(f" - Claims: {df_claims_clean.count()} rows")

StatementMeta(, 26b2b6d3-d14f-46c0-9f91-259bf09872a7, 3, Finished, Available, Finished)

Silver tables loaded:
 - Customers: 50 rows
 - Policies: 50 rows
 - Claims: 50 rows


---

## Join Customer

**Purpose:** Unite 3 tables to create a complete customer view (claim -> policy -> customer)

Multi-table joins and relationship mapping


In [4]:
df_joined = (
    df_claims_clean.alias("c")
    .join(
        df_policies_clean.alias("p"),
        col("c.policy_id") == col("p.policy_id"),
        "inner"
    )
    .join(
        df_customers_clean.alias("cu"),
        col("p.cust_id") == col("cu.cust_id"),
        "inner"
    )
)

print(f"Joined data: {df_joined.count()} rows")
display(df_joined.limit(5))


StatementMeta(, 26b2b6d3-d14f-46c0-9f91-259bf09872a7, 6, Finished, Available, Finished)

Joined data: 50 rows


SynapseWidget(Synapse.DataFrame, 4d310a3e-2c49-4eea-9eb0-629e166282bc)

---

## Claim Metrics
**Calculate claim metrics per customer**
- Aggregations (COUNT, SUM, AVG)

In [5]:
df_basic_metrics = df_joined.groupBy(
    "cu.cust_id",
    "cu.CustomerName",
    "cu.Gender"
).agg(
    count("c.claim_id").alias("TotalClaims"),
    round(sum("c.claim_amount"), 2).alias("TotalClaimAmount"),
    round(avg("c.claim_amount"), 2).alias("AvgClaimAmount")
)

print("Basic Claim Metrics:")
display(df_basic_metrics.orderBy(desc("TotalClaimAmount")).limit(10))

StatementMeta(, 26b2b6d3-d14f-46c0-9f91-259bf09872a7, 7, Finished, Available, Finished)

Basic Claim Metrics:


SynapseWidget(Synapse.DataFrame, b6ac5392-2900-4213-b9ca-bee6d34adcc7)

---

##  Claim Status Analysis

**Analyse claim approvals and rejections**
- Conditional logic (CASE WHEN) and rate calculations 

In [7]:
df_status_metrics = df_joined.groupBy(
    "cu.cust_id",
    "cu.CustomerName"
).agg(
    count(when(lower(col("c.status")) == "approved", True)).alias("ApprovedClaims"),
    count(when(lower(col("c.status")) == "rejected", True)).alias("RejectedClaims"),
    count("c.claim_id").alias("TotalClaims")
).withColumn(
    "ApprovalRate",
    round((col("ApprovedClaims") / col("TotalClaims")) * 100, 2)
)

print("Claim status analysis:")
display(df_status_metrics.orderBy(desc("ApprovalRate")).limit(10))

StatementMeta(, 26b2b6d3-d14f-46c0-9f91-259bf09872a7, 9, Finished, Available, Finished)

Claim status analysis:


SynapseWidget(Synapse.DataFrame, 45ca13be-fe54-44da-a895-9bd04c71493a)

---

## Policy Corerage Analysis

**Analyse relationship between policy coverage and claims**
- Aggregations and business ratio calculations

In [8]:
df_coverage_metrics = df_joined.groupBy(
    "cu.cust_id",
    "cu.CustomerName"
).agg(
    round(sum("p.coverage_amount"), 2).alias("TotalCoverageAmount"),
    round(sum("c.claim_amount"), 2).alias("TotalClaimAmount"),
    collect_set(lower(col("p.policy_type"))).alias("PolicyTypesArray")
).withColumn(
    "ClaimToCoverageRatio",
    round((col("TotalClaimAmount") / col("TotalCoverageAmount")) * 100, 2)
).withColumn(
    "PolicyTypes",
    concat_ws(", ", col("PolicyTypesArray"))
).drop("PolicyTypesArray")

print("Coverage Analysis:")
display(df_coverage_metrics.orderBy(desc("ClaimToCoverageRatio")).limit(10))

StatementMeta(, 26b2b6d3-d14f-46c0-9f91-259bf09872a7, 10, Finished, Available, Finished)

Coverage Analysis:


SynapseWidget(Synapse.DataFrame, bd2cc522-c9f5-443d-858f-5dff7a20d079)

---

## Temporal analysis

**Analyse temporal behavior of claims per customer**
- Date functions and temporal activity analysis

In [9]:
df_temporal_metrics = df_joined.groupBy(
    "cu.cust_id",
    "cu.CustomerName"
).agg(
    min("c.claim_date").alias("FirstClaimDate"),
    max("c.claim_date").alias("LastClaimDate"),
    count("c.claim_id").alias("TotalClaims")
).withColumn(
    "DaysBetweenFirstAndLast",
    datediff(col("LastClaimDate"), col("FirstClaimDate"))
)

print("Temporal Analysis:")
display(df_temporal_metrics.orderBy(desc("DaysBetweenFirstAndLast")).limit(10))

StatementMeta(, 26b2b6d3-d14f-46c0-9f91-259bf09872a7, 11, Finished, Available, Finished)

Temporal Analysis:


SynapseWidget(Synapse.DataFrame, fd6c3d3a-f624-417a-a07a-3d8c9040a1bb)

---

## Consolidate Gold Table

**Consolidate all metrics into final Gold Table**
- Integration of multiple aggregations into a dimensional model

In [10]:
df_gold_final = (
    df_joined.groupBy("cu.cust_id", "cu.CustomerName", "cu.Gender")
    .agg(
        # Claim counts
        count("c.claim_id").alias("TotalClaims"),
        count(when(lower(col("c.status")) == "approved", True)).alias("ApprovedClaims"),
        count(when(lower(col("c.status")) == "rejected", True)).alias("RejectedClaims"),

        # Claim amounts
        round(sum("c.claim_amount"), 2).alias("TotalClaimAmount"),
        round(avg("c.claim_amount"), 2).alias("AvgClaimAmount"),

        # Policy information
        collect_set(lower(col("p.policy_type"))).alias("PolicyTypesArray"),
        round(sum("p.coverage_amount"), 2).alias("TotalCoverageAmount"),

        # Temporal information
        min("c.claim_date").alias("FirstClaimDate"),
        max("c.claim_date").alias("LastClaimDate")
    )
    .withColumn("PolicyTypes", concat_ws(", ", col("PolicyTypesArray")))
    .withColumn(
        "ApprovalRate",
        round((col("ApprovedClaims") / col("TotalClaims")) * 100, 2)
    )
    .withColumn(
        "ClaimToCoverageRatio",
        round((col("TotalClaimAmount") / col("TotalCoverageAmount")) * 100, 2)
    )
    .drop("PolicyTypesArray")
)

print("Gold Table Preview:")
display(df_gold_final.orderBy(desc("TotalClaimAmount")).limit(10))

StatementMeta(, 26b2b6d3-d14f-46c0-9f91-259bf09872a7, 12, Finished, Available, Finished)

Gold Table Preview:


SynapseWidget(Synapse.DataFrame, ba6ccf2b-f80a-4aa7-b174-563cff69fa63)

---

## Save Gold Table

**Persist Gold table as Delta Lake**
- Save as Delta table in Lakehouse with schema merge


In [12]:
df_gold_final.write \
    .mode("overwrite") \
    .format("delta") \
    .option("mergeSchema", "true") \
    .saveAsTable("gold_customer_claims_analytics")

print("Gold table created: gold_customer_claims_analytics")
print(f" Total customers analyzed: {df_gold_final.count()}")
print(f" Columns: {len(df_gold_final.columns)}")
print(f"\nTable location: Tables/gold_customer_claims_analytics")

StatementMeta(, 26b2b6d3-d14f-46c0-9f91-259bf09872a7, 14, Finished, Available, Finished)

Gold table created: gold_customer_claims_analytics
 Total customers analyzed: 24
 Columns: 14

Table location: Tables/gold_customer_claims_analytics


---


## Data Quality Checks 
**Validate Gold table quality**
- Data quality best practices and validations


### Check 1: No nulls in key columns



In [14]:
print("Data Quality Checks:")
print("-" * 50)


null_checks = df_gold_final.select([
    count(when(col(c).isNull(), c)).alias(c)
    for c in ["cust_id", "CustomerName", "TotalClaims"]
])
print("\n1. Null values in key columns:")
display(null_checks)

StatementMeta(, 26b2b6d3-d14f-46c0-9f91-259bf09872a7, 16, Finished, Available, Finished)

Data Quality Checks:
--------------------------------------------------

1. Null values in key columns:


SynapseWidget(Synapse.DataFrame, eac53969-5a71-455d-9760-0dca5ca0ca41)

### Check 2: Valid ranges

In [15]:
print("\n2. Metric ranges:")
print(f"   Max TotalClaims: {df_gold_final.agg(max('TotalClaims')).collect()[0][0]}")
print(f"   Max TotalClaimAmount: ${df_gold_final.agg(max('TotalClaimAmount')).collect()[0][0]:,.2f}")
print(f"   Avg ApprovalRate: {df_gold_final.agg(avg('ApprovalRate')).collect()[0][0]:.2f}%")

StatementMeta(, 26b2b6d3-d14f-46c0-9f91-259bf09872a7, 17, Finished, Available, Finished)


2. Metric ranges:
   Max TotalClaims: 7
   Max TotalClaimAmount: $173,141.19
   Avg ApprovalRate: 22.65%


### Check 3: Distrubution per gender

In [16]:
print("\n3. Distribution by Gender:")
display(df_gold_final.groupBy("Gender").count().orderBy("Gender"))

print("\nAll quality checks passed")

StatementMeta(, 26b2b6d3-d14f-46c0-9f91-259bf09872a7, 18, Finished, Available, Finished)


3. Distribution by Gender:


SynapseWidget(Synapse.DataFrame, 801d0747-f7f1-49fb-8eb2-b5ce76090f38)


All quality checks passed


### Business insights summary
Generate executive insights for stakeholders
- Extract business value from data 

### Top 5 Customers by claim amount

In [17]:
print("Key Businee Insights")
print("=" * 70)


top_claimers = df_gold_final.orderBy(desc("TotalClaimAmount")).limit(5)
print("\n1. TOP 5 CUSTOMERS BY CLAIM AMOUNT:")
display(top_claimers.select("CustomerName", "TotalClaimAmount", "TotalClaims", "ApprovalRate"))

StatementMeta(, 26b2b6d3-d14f-46c0-9f91-259bf09872a7, 19, Finished, Available, Finished)

Key Businee Insights

1. TOP 5 CUSTOMERS BY CLAIM AMOUNT:


SynapseWidget(Synapse.DataFrame, 6cccfc5f-9fd5-4c3c-a9d4-009170b7b0de)

### Approval rate distribution

In [18]:

approval_buckets = df_gold_final.withColumn(
    "ApprovalBucket",
    when(col("ApprovalRate") >= 80, "High (80%+)")
    .when(col("ApprovalRate") >= 50, "Medium (50-79%)")
    .otherwise("Low (<50%)")
).groupBy("ApprovalBucket").count().orderBy("ApprovalBucket")
display(approval_buckets)

StatementMeta(, 26b2b6d3-d14f-46c0-9f91-259bf09872a7, 20, Finished, Available, Finished)

SynapseWidget(Synapse.DataFrame, d520bfa4-f1f3-4630-a625-f865918c31f8)

### High risk customers (claim/coverage > 50%)

In [21]:
high_risk = df_gold_final.filter(col("ClaimToCoverageRatio") > 50).count()
print(f" {high_risk} customers with high claim-to-coverage ratio")


StatementMeta(, 26b2b6d3-d14f-46c0-9f91-259bf09872a7, 23, Finished, Available, Finished)

 7 customers with high claim-to-coverage ratio
